# Flatten countries and oceans shapefile

Generate a shapefile with only land and ocean.


### Input data:
Processed in `generate_shp_countries_ne_10m.ipynb` and `generate_shp_oceans_ne_10m.ipynb`

**Natural Earth**:

Admin 0, Countries — [Natural Earth 1:10m Cultural Vectors](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/)

Oceans — [Natural Earth 1:10m Physical Vectors](https://www.naturalearthdata.com/downloads/10m-physical-vectors/)

### Input data license:

CC0 1.0 Universal (public domain)

Giulia Cigna - giulia.cigna@polito.it<br>
Romain Thomas - romain.thomas@polito.it<br>
2026

In [1]:
import os
import chardet
import numpy as np
from dotenv import load_dotenv
import geopandas as gpd
import pandas as pd
import logging
from pathlib import Path

## LOGGING

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)

## SETTINGS

In [3]:
if not os.path.exists(".env"):
    raise ValueError("You must create the '.env' file and set the values before running this notebook.")

load_dotenv()

True

## OUTPUT SETTINGS

In [4]:
# Get full shapefile path from environment
output_file = os.getenv("NE_LAND_OCEAN_PATH")
if output_file is None:
    raise ValueError("NE_LAND_OCEAN_PATH environment variable is not set")

full_path = Path(output_file)

# Create parent directory if it doesn't exist
full_path.parent.mkdir(parents=True, exist_ok=True)

logging.info(f"Output path: {full_path}")


2026-03-24 12:31:39 - root - INFO - Output path: data\ne_10m_land_ocean\ne_10m_land_ocean.shp


## INPUT FILES

In [5]:
countries_from_ne_path = os.getenv("COUNTRIES_FROM_NE_PATH")
if countries_from_ne_path is None:
    raise ValueError("COUNTRIES_FROM_NE_PATH environment variable is not set")

oceans_from_ne_path = os.getenv("NE_GEO_OCEAN_PATH")
if oceans_from_ne_path is None:
    raise ValueError("NE_GEO_OCEAN_PATH environment variable is not set")

# CSV with regions ID
codes_id_path = os.getenv('CODES_ID_PATH')
if codes_id_path is None:
    raise ValueError("CODES_ID_PATH environment variable is not set")

## READING INPUT FILES

In [6]:
# countries
logging.info(f"Reading countries shapefile  from {countries_from_ne_path}")
gdf_countries = gpd.read_file(countries_from_ne_path)

# oceans
logging.info(f"Reading ocean shapefile from {oceans_from_ne_path}")
gdf_oceans = gpd.read_file(oceans_from_ne_path)

2026-03-24 12:31:39 - root - INFO - Reading countries shapefile  from data/countries_from_ne_10m/countries_from_ne_10m.shp
2026-03-24 12:31:39 - root - INFO - Reading ocean shapefile from data/ne_10m/ne_10m_ocean/ne_10m_ocean.shp


In [7]:
# regions ID
logging.info(f"Reading ids from {codes_id_path}")
with open(codes_id_path, "rb") as f:
    result = chardet.detect(f.read())

# from https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html#na-values
na_vals = ['-1.#IND', '1.#QNAN', '1.#IND', '-1.#QNAN', '#N/A N/A', '#N/A', 'N/A', 'n/a', 'NA', '<NA>', '#NA', 'NULL', 'null', 'NaN', '-NaN', 'nan', '-nan', 'None', '']
# avoids errors with country code "NA":
na_vals.remove('NA')

codes_id = pd.read_csv(
    codes_id_path,
    encoding=result["encoding"],
    sep=None,
    engine="python",
    keep_default_na=False,
    na_values=na_vals
)

2026-03-24 12:31:39 - root - INFO - Reading ids from data/codes_id.csv


## ALIGNMENT CHECK

In [8]:
# Verify columns alignment

if gdf_countries.columns.all() == gdf_oceans.columns.all():
    logging.info("Columns aligned")
else:
    logging.info("Columns system not aligned")


# Verify reference system alignment
logging.info(f"Countries reference system: {gdf_countries.crs}")
logging.info(f"Oceans reference system: {gdf_oceans.crs}")


if gdf_countries.crs == gdf_oceans.crs:
    logging.info("Reference system aligned")
else:
    logging.info("Reference system not aligned")

2026-03-24 12:31:40 - root - INFO - Columns aligned
2026-03-24 12:31:40 - root - INFO - Countries reference system: EPSG:4326
2026-03-24 12:31:40 - root - INFO - Oceans reference system: EPSG:4326
2026-03-24 12:31:40 - root - INFO - Reference system aligned


## FLATTEN COUNTRIES INTO LAND

In [9]:
# Drop useless columns
gdf_countries = gdf_countries.drop(columns=["ISO3_CODE", "ISO2_CODE", "ISON_CODE", "NE_ID"])

# Merge geometries in a single one
gdf_land = gdf_countries.dissolve()

# Metadata
gdf_land["NAME"] = "Land"
gdf_land["ID"] = "L"


## SINGLE OCEAN METADATA

In [10]:
# Drop useless columns
gdf_oceans = gdf_oceans.drop(columns=["scalerank", "min_zoom", "featurecla"])

# Merge geometries in a single one
gdf_ocean = gdf_oceans.dissolve()

# Metadata
gdf_ocean["NAME"] = "Ocean"
gdf_ocean["ID"] = "OC"
gdf_ocean["SOURCE"] = "Natural Earth Ocean"
gdf_ocean["NUM_ID"] = np.nan


## "NUM_ID" FROM codes_id.csv

In [11]:
# Add the NUM_ID from the csv file
lookup = codes_id.set_index("ID")["NUM_ID"]

gdf_land["NUM_ID"] = gdf_land["ID"].map(lookup)
gdf_land["NUM_ID"] = gdf_land["NUM_ID"].astype(str).str.strip()
gdf_land["NUM_ID"] = pd.to_numeric(gdf_land["NUM_ID"], errors='coerce').astype('Int64')

gdf_ocean["NUM_ID"] = gdf_ocean["ID"].map(lookup)
gdf_ocean["NUM_ID"] = gdf_ocean["NUM_ID"].astype(str).str.strip()
gdf_ocean["NUM_ID"] = pd.to_numeric(gdf_ocean["NUM_ID"], errors='coerce').astype('Int64')


## MERGING

In [12]:
# Merge together the gdfs
gdf_final = gpd.GeoDataFrame(
    pd.concat([gdf_land, gdf_ocean], ignore_index=True),
    crs="EPSG:4326"
)

## SAVE OUTPUT

In [13]:
# Save GeoDataFrame
gdf_final.to_file(full_path)

logging.info(f"Saved Shapefile to: {full_path}")

2026-03-24 12:31:44 - pyogrio._io - INFO - Created 2 records
2026-03-24 12:31:44 - root - INFO - Saved Shapefile to: data\ne_10m_land_ocean\ne_10m_land_ocean.shp
